## Databricks DQX - Data Quality Checks
This notebook demonstrates how to use [Databricks Labs DQX](https://github.com/databrickslabs/dqx) to:
1. **Profile** a table to understand its data characteristics
2. **Generate** data quality rules automatically based on the profile
3. **Store** the generated rules in a Delta table for reuse
4. **Load and apply** those rules to validate data quality
5. **Optional: Store cleaned data to table** save the cleaned data to a silver table

### Step 1: Install DQX Library
Install the `databricks-labs-dqx` package and restart the Python environment to make it available for subsequent cells.

In [0]:
# Install the Databricks Labs DQX library for data quality profiling and validation
%pip install 'databricks-labs-dqx'

# Restart Python interpreter to pick up the newly installed package
%restart_python

### Step 2: Configure Parameters
Set up notebook widgets to allow dynamic configuration of the target catalog, schema, and table name. These widgets appear at the top of the notebook for easy runtime changes.

In [0]:
# Define default values for the target table location
default_catalog = "tenbosch"
default_schema = "adventureworks"
default_table = "dim_customer"

# Create notebook widgets for dynamic parameter input at runtime
dbutils.widgets.text("demo_catalog", default_catalog, "Catalog Name")
dbutils.widgets.text("demo_schema", default_schema, "Schema Name")
dbutils.widgets.text("demo_table", default_table, "Table Name")


### Step 3: Load the Target Table
Construct the fully qualified table name from widget parameter values and load a preview of the data to verify connectivity.

In [0]:
# Retrieve widget values for catalog, schema, and table
demo_catalog_name = dbutils.widgets.get("demo_catalog")
demo_schema_name = dbutils.widgets.get("demo_schema")
demo_table_name = dbutils.widgets.get("demo_table")

# Construct the fully qualified table name (catalog.schema.table)
full_table_name = demo_catalog_name + "." + demo_schema_name + "." + demo_table_name

# Load the table as a Spark DataFrame and display a 10-row sample
table_to_profile = spark.table(full_table_name)
display(table_to_profile.limit(10))

### Step 4: Profile the Table
Run the DQX profiler to generate summary statistics and per-column profiles. These profiles capture data distributions, null rates, and value ranges that inform rule generation.

In [0]:
# Import WorkspaceClient for authenticated access to Databricks APIs
from databricks.sdk import WorkspaceClient
# Import DQProfiler to analyze table columns and generate statistical profiles
from databricks.labs.dqx.profiler.profiler import DQProfiler

# Initialize the WorkspaceClient (uses the notebook's built-in authentication)
ws = WorkspaceClient()

# Create a profiler instance and run it against the target table
# Returns summary_stats (high-level table metrics) and profiles (per-column details)
profiler = DQProfiler(ws)
summary_stats, profiles = profiler.profile(table_to_profile)

# Uncomment to inspect the raw profile output
#print(profiles)

### Step 5: Generate Data Quality Rules
Use the `DQGenerator` to automatically create data quality check rules from the column profiles. Rules include null checks, range validations, and pattern matching.

In [0]:
# Import the DQGenerator to create rules from profile results
from databricks.labs.dqx.profiler.generator import DQGenerator

# Generate data quality check rules automatically from column profiles
generator = DQGenerator(WorkspaceClient())
generated_rules = generator.generate_dq_rules(profiles)

# Uncomment to view the generated rules in YAML format
#print(generated_rules)


### Step 6: Save Rules to Delta Table
Persist the generated rules to a Delta table for versioning, sharing, and reuse across pipelines.

In [0]:
# Import storage config and DQ engine for persisting and running checks
from databricks.labs.dqx.config import TableChecksStorageConfig
from databricks.labs.dqx.engine import DQEngine

# Define the Delta table location for storing DQ rules (appends "_dq_checks" suffix)
rules_table = f"{demo_catalog_name}.{demo_schema_name}.{demo_table_name}_dq_checks"

# Initialize the DQ engine with the authenticated workspace client
dq_engine = DQEngine(ws)

# Save the generated rules to a Delta table, overwriting any existing rules
dq_engine.save_checks(generated_rules, config=TableChecksStorageConfig(location=rules_table, mode="overwrite"))

# Display the saved rules for verification
display(spark.table(rules_table))


### Step 7: Load and Apply Rules
Load the saved rules from the Delta table and apply them against a sample of the target data. The engine appends validation result columns (`_warnings` and `_errors`) to each row.

In [0]:
# Load the previously saved DQ rules from the Delta table
loaded_rules = dq_engine.load_checks(config=TableChecksStorageConfig(location=rules_table))

# Load a sample of 100 rows from the target table for validation
table_to_dq_check = spark.table(full_table_name).limit(100)

# Apply the loaded rules to the DataFrame; appends _warnings and _errors columns
checked_df = dq_engine.apply_checks_by_metadata(table_to_dq_check, loaded_rules)


### Step 8: Display Validation Results
Show the final DataFrame with data quality check results. Rows that violate rules will have details in the appended `_warnings` and `_errors` columns.

In [0]:
# Display the final DataFrame with DQ validation results (_warnings and _errors columns)
display(checked_df)

### Step 9: Split Valid and Quarantined Data
Apply DQ rules and split the result into two DataFrames: rows that pass all checks (clean data) and rows that fail one or more checks (quarantined data). Display the quarantined rows for review.

In [0]:
# Apply rules and split into clean rows vs. rows that failed validation
cleaned_table, quarantine_data = dq_engine.apply_checks_by_metadata_and_split(table_to_dq_check, loaded_rules)

# Display only the quarantined (failed) rows for investigation
display(quarantine_data)

### Step 10: Save Clean Data to Silver Table
Optionally persist the validated (clean) DataFrame to a Delta silver table for downstream consumption.

In [0]:
# Write the validated DataFrame to a Delta silver table (overwrites if it already exists)
table_to_dq_check.write.format("delta").mode("overwrite").saveAsTable(
    f"{demo_catalog_name}.{demo_schema_name}.{demo_table_name}_silver"
)